#### hello 도구 테스트

In [ ]:
# https://github.com/langchain-ai/langchain-mcp-adapters?tab=readme-ov-file#streamable-http

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

from langchain_mcp_adapters.tools import load_mcp_tools # uv add langchain-mcp-adapters

async with streamablehttp_client("http://127.0.0.1:8000/mcp/") as (read, write, _):
    async with ClientSession(read, write) as session:
        await session.initialize()

        # Get tools
        tools = await load_mcp_tools(session)

        # 'hello' 도구를 비동기적으로 실행하고 결과를 result 변수에 할당
        # MCP는 서버이기 때문에 네트워크로 비동기 통신함
        result = await tools[0].ainvoke({"name": "김일남"})
        print(result)

[{'type': 'text', 'text': '안녕하세요, 김일남님!', 'id': 'lc_2885e198-621f-4db6-b313-94c55d660e94'}]


#### 랭그래프에서 MCP 연동

> Use MCP: <https://langchain-ai.github.io/langgraph/agents/mcp/#use-mcp-tools> </br>
> GitHub: <https://github.com/langchain-ai/langchain-mcp-adapters?tab=readme-ov-file#using-with-langgraph-stategraph>

In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "mcp_tools": {
            "url": "http://localhost:8000/mcp",
            "transport": "streamable_http",
        }
    }
)
tools = await client.get_tools()

In [3]:
print(f"로드된 도구들: {[t.name for t in tools]}") 

로드된 도구들: ['hello', 'get_current_time', 'get_yf_stock_history', 'get_web_search']


In [5]:
tools

[StructuredTool(name='hello', description='간단한 인사말을 반환하는 도구', args_schema={'properties': {'name': {'default': '아무개', 'title': 'Name', 'type': 'string'}}, 'title': 'helloArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x000001D819ABE8E0>),
 StructuredTool(name='get_current_time', description=" 현재 시각을 반환하는 함수\n\n    Args:\n        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함\n        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨\n    ", args_schema={'properties': {'timezone': {'default': 'Asia/Seoul', 'title': 'Timezone', 'type': 'string'}, 'location': {'default': '부산', 'title': 'Location', 'type': 'string'}}, 'title': 'get_current_timeArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x000001D819ABE980>),
 StructuredTool(name='get_yf_stock_history', description=' 주식

In [6]:
# 에이전트 생성 전 출력해보기
print(f"로드된 도구 개수: {len(tools)}")
for t in tools:
    print(f"- {t.name}: {t.description}")

로드된 도구 개수: 4
- hello: 간단한 인사말을 반환하는 도구
- get_current_time:  현재 시각을 반환하는 함수

    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨
    
- get_yf_stock_history:  주식 종목의 가격 데이터를 조회하는 함수
- get_web_search: 
    특정 지역과 기간을 설정하여 웹 검색(뉴스)을 수행합니다.

    Args:
        query (str): 검색어
        search_period (str): 검색 기간. 'd'(일간), 'w'(주간), 'm'(월간), 'y'(연간) 중 선택
        region (str): 검색 지역 코드. 한국('kr-kr'), 미국('us-en'), 일본('jp-jp'), 영국('uk-en') 등
    
    Returns:
        str: 검색된 뉴스 결과 리스트
    


In [7]:
result = await tools[0].ainvoke({"name": "김일남"})
result

[{'type': 'text',
  'text': '안녕하세요, 김일남님!',
  'id': 'lc_6f9f24b8-c2ad-4cdd-9430-f630b0a441c3'}]

In [8]:
result = await tools[1].ainvoke({"timezone": "Asia/Seoul", "location": "부산"})
result

[{'type': 'text',
  'text': 'Asia/Seoul (부산) 현재시각 2026-02-01 21:01:39 ',
  'id': 'lc_2f5ebe5d-dd87-46e0-ad83-ea0698bd318f'}]

In [9]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

In [10]:
from dataclasses import dataclass

@dataclass
class UserContext:
    user_id: str

In [11]:
system_prompt = (
    "너는 사용자의 질문에 답변을 하기 위해 도구를 사용할 수 있는 비서다. "
    "지역에 대한 시간을 물어보면, 해당 지역의 타임존(예: 서울/부산은 'Asia/Seoul')을 "
    "스스로 추론하여 get_current_time 도구를 즉시 호출해라."
)

In [12]:
from langchain.agents import create_agent

# 에이전트 생성
agent = create_agent(
    model,
    tools=tools,
    context_schema=UserContext,
    system_prompt=system_prompt
)

In [13]:
# 비동기 호출 요망
result = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "부산은 지금 몇시야?"}]},
    context=UserContext(user_id="user123")
)

In [14]:
result

{'messages': [HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}, id='4bc1e55b-a9e0-48f0-a7a4-c775b9715b41'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_current_time', 'arguments': '{"timezone": "Asia/Seoul", "location": "\\ubd80\\uc0b0"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--b065db30-3686-41fb-abbf-134a6eb5b49b-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'f2a1f619-2a0f-4fa2-9e17-26f550db25ed', 'type': 'tool_call'}], usage_metadata={'input_tokens': 570, 'output_tokens': 24, 'total_tokens': 594, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content=[{'type': 'text', 'text': 'Asia/Seoul (부산) 현재시각 2026-02-01 21:01:57 ', 'id': 'lc_daaca072-67f7-427f-8311-edf04b2cbafd'}], name='get_cu